# Transfer Learning with MobileNetV2

This notebook adapts ImageNet features to a small binary image task. The convolutional base stays frozen while a new classification head learns the task-specific labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

tf.random.set_seed(42)

# MNIST supplies a compact, reproducible task; duplicate the grayscale channel
# so it can be consumed by an ImageNet-pretrained feature extractor.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
keep_train = y_train < 2
keep_test = y_test < 2
x_train, y_train = x_train[keep_train][:4000], y_train[keep_train][:4000]
x_test, y_test = x_test[keep_test][:800], y_test[keep_test][:800]

def prepare_images(images):
	images = tf.cast(images[..., None], tf.float32)
	images = tf.image.grayscale_to_rgb(images)
	images = tf.image.resize(images, (96, 96))
	return preprocess_input(images)

x_train = prepare_images(x_train)
x_test = prepare_images(x_test)

base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(96, 96, 3))
base.trainable = False
model = models.Sequential([
	base,
	layers.GlobalAveragePooling2D(),
	layers.Dropout(0.2),
	layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, validation_split=0.1, epochs=2, batch_size=64, verbose=1)
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")

plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()